# Docking Simulation

This notebook performs molecular docking of the prepared ligands into the receptor binding pocket using **AutoDock Vina**.
It iterates through the prepared PDBQT files, runs Vina, parses the binding affinity, and aggregates the results.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import os
import subprocess
import glob
import shutil
from datetime import datetime
import multiprocessing

## Configuration

Setup paths and create a timestamped directory for the docking run results.

In [2]:
max_cpu = multiprocessing.cpu_count()
print(f"Available CPUs: {max_cpu}")

In [3]:
if input("Are you sure? [y/n] ").lower() == "y":
    VINA_PATH = "vina"
    RECEPTOR_PATH = Path("..") / "data" / "raw" / "3FXI.pdbqt"
    LIGANDS_FOLDER = Path("..") / "data" / "raw" / "ligands"

    DOCKING_FOLDER = Path("..") / "data" / "docking" / f"""run_{datetime.now().strftime("%Y-%m-%d")}"""
    BEST_CSV_PATH = Path("..") / "data" / "processed" / "best_sepsis_candidates.csv"

    if DOCKING_FOLDER.exists():
        shutil.rmtree(DOCKING_FOLDER)

    DOCKING_FOLDER.mkdir(parents=True, exist_ok=True)

    DOCKING_OUTPUT_FOLDER = DOCKING_FOLDER / "output"
    DOCKING_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
else:
    raise Exception("Aborted by user")

## Grid Parameters

Define the binding pocket coordinates (center and size) for the docking simulation.

In [4]:
CENTER_X = -3
CENTER_Y = -9
CENTER_Z = -27
 
SIZE_X = 60
SIZE_Y = 60
SIZE_Z = 60

EXHAUSTIVENESS = 16 

## Execute Docking

Loop through all `.pdbqt` ligand files:
1.  Construct Vina command.
2.  Run Vina via `subprocess`.
3.  Parse individual binding affinities from `stdout`.
4.  Merge results with candidate data and save to `final_docking_scores.csv`.

In [5]:
results_list = []

ligand_files = glob.glob(os.path.join(LIGANDS_FOLDER, "*.pdbqt"))

print(f"Found ligands: {len(ligand_files)}")

for i, ligand_path in enumerate(ligand_files):
    ligand_name = Path(ligand_path).stem
    output_file = DOCKING_OUTPUT_FOLDER / f"{ligand_name}_out.pdbqt"
    log_file = DOCKING_FOLDER / f"{ligand_name}.log"

    print(f"[{i+1:02}/{len(ligand_files):02}] Docking: {ligand_name}...")

    # Use fewer CPUs per job to allow system stability, or use max_cpu - 2
    cpus_to_use = max(1, max_cpu - 2)

    command = [
        VINA_PATH,
        '--receptor', str(RECEPTOR_PATH),
        '--ligand', str(ligand_path),
        '--out', str(output_file),
        '--center_x', str(CENTER_X),
        '--center_y', str(CENTER_Y),
        '--center_z', str(CENTER_Z),
        '--size_x', str(SIZE_X),
        '--size_y', str(SIZE_Y),
        '--size_z', str(SIZE_Z),
        '--exhaustiveness', str(EXHAUSTIVENESS),
        '--cpu', str(cpus_to_use)
    ]

    try:
        result = subprocess.run(command, capture_output=True, text=True)
        
        best_affinity = None
        
        # Parse Vina output for the best affinity (first mode)
        for line in result.stdout.splitlines():
            if line.strip().startswith('1'):
                parts = line.split()
                if len(parts) >= 2:
                    best_affinity = float(parts[1])
                    break
        
        if best_affinity is not None:
            results_list.append({
                'Ligand': ligand_name,
                'Affinity': best_affinity,
                "COMB": "_".join(ligand_name.split("_")[2:])
            })
        else:
            print(f"  Warning: Could not parse energy for {ligand_name}")
            with open(log_file, 'w') as f:
                f.write(result.stderr)

    except Exception as e:
        print(f"Execution error: {e}")

# --- Save Results ---
if results_list:
    df_res = pd.DataFrame(results_list)
    df_best = pd.read_csv(BEST_CSV_PATH)
    df = pd.merge(
        df_res,
        df_best,
        on="COMB",
        how="inner"
    )
    df = df.sort_values(by='Affinity') # Lower is better
    df.to_csv(DOCKING_FOLDER / 'final_docking_scores.csv', index=False)
    print("\nDocking Completed!")
    print("Top 5 Results:")
    print(df.head())
else:
    print("No results generated. Check paths and Vina installation.")